# 06 — Inference with your distilled model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenTracy/OpenTracy/blob/main/notebooks/06_distilled_inference.ipynb)

You trained a student in notebook 05. Now **use it every day in production**. This notebook walks through three serving paths — from "load the LoRA in a Python script" to "swap it behind a routing alias so your app code never changes" — and closes with how to tell when to retrain.

In this notebook:

1. **Load the student** with `opentracy.Student` — smallest possible serving
2. *(optional)* **Serve via llama.cpp** (GGUF) — CPU-only, edge-ready
3. **The alias swap** — register under a name, call it via `ot.completion(model="...")`, re-point as needed
4. **Monitor drift** — when to retrain

## Prerequisites

Run notebook 05 first. It prints a line like:

```
✓ trained student: backend=peft base=unsloth/Llama-3.2-1B-Instruct-bnb-4bit
   artifact at: /tmp/opentracy-distill-XXXXXX/distillation/local-XXXXXXXX/adapter
```

Copy that `artifact at:` path into `ADAPTER_PATH` two cells down. If notebook 05 also exported a GGUF, grab that path too. If not (e.g. no `llama.cpp` on the host), `ot.distill()` falls back to the LoRA adapter and you can still do everything here.

> **Runtime:** GPU preferred for the PEFT path (it loads the base model). CPU is enough for the GGUF path.

## Setup

In [30]:
!pip install opentracy transformers peft accelerate -q
# Optional — only needed if you want the GGUF / llama.cpp path below.
# !pip install llama-cpp-python -q

In [31]:
import os
from getpass import getpass

# Only needed for the drift-monitoring section (shadow sampling vs GPT-4o).
# Sections 1–3 run entirely local and don't need a key.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY (optional, only for drift section): ")
    except Exception:
        pass


In [32]:
# ← Paste the path that notebook 05 printed after `✓ trained student`
ADAPTER_PATH = "/tmp/opentracy-distill-xo37njz5/distillation/local-1d88a9fd/adapter"

# Optional — only set if notebook 05 actually exported a GGUF.
# If phase 4 of nb05 skipped GGUF export (no llama.cpp on host), leave this as None.
GGUF_PATH = None

import pathlib
assert pathlib.Path(ADAPTER_PATH).exists(), (
    f"Adapter not found at {ADAPTER_PATH}. Run notebook 05 first and paste its "
    "`artifact at:` path here."
)
print("using adapter at:", ADAPTER_PATH)
if GGUF_PATH:
    assert pathlib.Path(GGUF_PATH).exists(), f"GGUF not found at {GGUF_PATH}"
    print("using GGUF at:   ", GGUF_PATH)
else:
    print("(no GGUF path set — the llama.cpp section will be skipped)")

using adapter at: /tmp/opentracy-distill-xo37njz5/distillation/local-1d88a9fd/adapter
(no GGUF path set — the llama.cpp section will be skipped)


## 1. Serve via `opentracy.Student`

`Student` is a thin wrapper around the adapter or GGUF file. Call it as a function or feed it to `ot.completion()` — no HTTP server, no provider call, just local inference.

For a LoRA-only adapter, `Student` loads the base model once (~1 GB first time from HuggingFace, cached afterwards) and attaches the adapter on top.

In [33]:
from opentracy import Student

student = Student(
    backend="peft",
    model_path=ADAPTER_PATH,
    base_model="unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
)
print("student ready")

student ready


In [34]:
TEMPLATE = "Classify this ticket (billing/technical/feature_request/other): '{t}'"

for ticket in [
    "My invoice for March hasn't arrived",
    "Getting 500 errors when calling /v1/users/me",
    "Please add a dark mode",
]:
    label = student(TEMPLATE.format(t=ticket), max_new_tokens=10).strip()
    print(f"[{label:<18}]  {ticket}")

[This ticket should be classified as "billing."]  My invoice for March hasn't arrived
[This ticket should be classified as "technical" since]  Getting 500 errors when calling /v1/users/me
[This ticket should be classified as a "feature_request]  Please add a dark mode


## 2. *(optional)* Serve via llama.cpp (GGUF)

For edge deployments, on-device inference, or when you don't want Python in the serving path, use the GGUF file produced by notebook 05 — if you have one. Skipped automatically if `GGUF_PATH` above is `None`.

In [35]:
if GGUF_PATH:
    from llama_cpp import Llama
    llm = Llama(model_path=GGUF_PATH, n_ctx=2048, n_threads=4, verbose=False)

    for ticket in [
        "How do I update my billing address?",
        "Rate limit header says 60/min but we're being throttled at 30",
    ]:
        resp = llm.create_chat_completion(
            messages=[{"role": "user", "content": TEMPLATE.format(t=ticket)}],
            max_tokens=10, temperature=0,
        )
        label = resp["choices"][0]["message"]["content"].strip()
        print(f"[{label:<18}]  {ticket}")
else:
    print("Skipped — GGUF_PATH is not set (no GGUF was exported in nb05).")

Skipped — GGUF_PATH is not set (no GGUF was exported in nb05).


The 4-bit GGUF file is ~500 MB and runs on any CPU. Common deployment targets:

- **llama.cpp server** — `./server -m model.q4_k_m.gguf -c 2048`
- **Ollama** — `ollama create ticket-classifier -f Modelfile`
- **LM Studio** — drop the file in the models folder
- **On-device mobile** — llama.cpp compiles on iOS / Android

## 3. The alias swap — the closing move

In production you don't want your app aware of adapter paths or GGUF files. You want it to call `model="ticket-classifier"` and forget. `student.deploy(alias)` writes the mapping into `~/.opentracy/aliases.json` — from that point on, `ot.completion(model=alias, ...)` dispatches locally to this Student.

**Your app code doesn't change when the alias's target changes.** That's the piece people don't see coming.

In [36]:
import opentracy as ot

entry = student.deploy("ticket-classifier")
print("registered:", entry)

# Now ot.completion(model="ticket-classifier", ...) routes here — zero HTTP, zero API spend.
resp = ot.completion(
    model="ticket-classifier",
    messages=[{"role": "user", "content": TEMPLATE.format(t="Cancel my subscription")}],
    max_tokens=10,
    temperature=0,
)
print("answer:", resp.choices[0].message.content.strip())
print(f"cost:   ${resp._cost:.8f}  (served locally — no provider fee)")

registered: {'backend': 'peft', 'model_path': '/tmp/opentracy-distill-xo37njz5/distillation/local-1d88a9fd/adapter', 'base_model': 'unsloth/Llama-3.2-1B-Instruct-bnb-4bit', 'metadata': {}, 'registered_at': '2026-04-22T01:25:13Z'}
answer: This ticket should be classified as "billing" since
cost:   $0.00000000  (served locally — no provider fee)


### Re-pointing an alias

Say your app has been shipping with `model="smart"` aliased to `openai/gpt-4o`. When the student is ready, re-point `smart` to the local student:

```python
ot.set_alias("smart", backend="peft", model_path=ADAPTER_PATH, base_model="unsloth/Llama-3.2-1B-Instruct-bnb-4bit")
```

Rolling back is the same call with different args (or `ot.unset_alias("smart")` to drop the mapping entirely, which makes `"smart"` route back to whatever the provider resolution decides). Below we just show the registered entry.

In [37]:
# Show what's in the alias registry right now
for name, entry in ot.list_aliases().items():
    print(f"  {name:<24} → {entry['backend']:<6} {entry['model_path']}")

  ticket-classifier        → peft   /tmp/opentracy-distill-xo37njz5/distillation/local-1d88a9fd/adapter


## 4. Serve it as an OpenAI-compatible HTTP endpoint

Everything above runs inside one Python process. For a network-accessible endpoint that other services can hit, wrap the alias in ~15 lines of FastAPI. Anything that speaks the OpenAI `/v1/chat/completions` wire format — LangChain, LlamaIndex, a curl script, a mobile SDK — works against it unchanged.

The cell below writes `serve.py` to disk and prints the launch command. Run it in a second terminal (it blocks) — we don't start it inline because uvicorn would hang the notebook kernel.


In [ ]:
import pathlib, sys, textwrap

SERVE_PY = '''\
"""Minimal OpenAI-compatible FastAPI server backed by an OpenTracy alias.

Surfaces:
  GET  /health                    -> {"status":"ok","aliases":[...]}
  GET  /v1/models                 -> OpenAI model list (drawn from registered aliases)
  POST /v1/chat/completions       -> OpenAI-compatible completion
"""
from __future__ import annotations

import logging
import traceback
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel
import opentracy as ot

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("opentracy-serve")

app = FastAPI(title="OpenTracy Local Serve")


class ChatRequest(BaseModel):
    model: str = "ticket-classifier"   # alias registered via student.deploy()
    messages: list
    max_tokens: int = 64
    temperature: float = 0.0


@app.get("/health")
def health():
    return {"status": "ok", "aliases": list(ot.list_aliases().keys())}


@app.get("/v1/models")
def models():
    return {"data": [{"id": name, "object": "model"} for name in ot.list_aliases()]}


@app.post("/v1/chat/completions")
def complete(req: ChatRequest):
    try:
        resp = ot.completion(
            model=req.model,
            messages=req.messages,
            max_tokens=req.max_tokens,
            temperature=req.temperature,
        )
    except Exception as e:
        logger.exception("completion failed")
        raise HTTPException(
            status_code=500,
            detail={"error": f"{type(e).__name__}: {e}", "traceback": traceback.format_exc()},
        )
    # ModelResponse is a dict subclass; force JSONResponse so underscore-prefixed
    # extras (_cost, _latency_ms, _provider, _routing) are preserved on the wire.
    return JSONResponse(content=dict(resp))
'''

pathlib.Path("serve.py").write_text(SERVE_PY)
print(f"wrote serve.py ({len(SERVE_PY)} bytes)\n")

# Use THIS kernel's Python — otherwise a system `uvicorn` on PATH can pick up a
# different interpreter (e.g. py3.10 with stale jinja2 / no opentracy install)
# and every request 500s. sys.executable is always the Python that's running now.
#
# NOTE: curl lines are deliberately on ONE line — copy-pasting a backslash
# line-continuation into a terminal with trailing whitespace silently breaks
# the command (`-d: command not found`), and that error has eaten debugging
# hours for more than one user.
body = '{"model":"ticket-classifier","messages":[{"role":"user","content":"Classify: refund please"}],"max_tokens":10}'
launch = textwrap.dedent(f"""
    # 1. install server deps in THIS kernel's Python (once)
    {sys.executable} -m pip install -U "fastapi" "uvicorn" "jinja2>=3.1"

    # 2. launch in a second terminal (blocks; watch stdout for errors)
    {sys.executable} -m uvicorn serve:app --host 0.0.0.0 --port 9000

    # 3. sanity-check it's up
    curl -s http://localhost:9000/health

    # 4. hit the completions endpoint (single line — don't add backslashes)
    curl -sS -X POST http://localhost:9000/v1/chat/completions -H 'Content-Type: application/json' -d '{body}'
""").strip()
print(launch)


## 5. Monitor drift — when to retrain

Traffic shifts. The prompts you trained on may not match next quarter's prompts. Two signals to watch:

1. **Student-vs-teacher agreement on new traffic.** Sample N% of requests, ask the teacher in parallel, compare labels.
2. **Cluster distribution shift.** If traffic moves into clusters the student wasn't trained for, the router's error estimates climb.

The loop below shadows 10 fresh tickets through both and prints agreement + cost.


In [ ]:
import os

shadow_tickets = [
    "Credit card declined on renewal — please help",
    "Need webhook delivery retries with exponential backoff",
    "Add a cron-based scheduled export",
    "What region is your data stored in?",
    "Account locked out after failed 2FA",
    "Please upgrade my plan to annual",
    "GraphQL endpoint returning 502 intermittently",
    "Invoice PDF missing line items",
    "Support OAuth device flow?",
    "When is the next product webinar?",
]

# Teacher needs OPENAI_API_KEY OR an OPENTRACY_ENGINE_URL that has one.
have_teacher = bool(os.environ.get("OPENAI_API_KEY") or os.environ.get("OPENTRACY_ENGINE_URL"))
if not have_teacher:
    print("Skipped: no OPENAI_API_KEY set (and no OPENTRACY_ENGINE_URL). Classify-only below.")
    for t in shadow_tickets:
        s = ot.completion(model="ticket-classifier", messages=[{"role":"user","content":TEMPLATE.format(t=t)}],
                          max_tokens=10, temperature=0)
        print(f"  [{s.choices[0].message.content.strip():<18}]  {t}")
else:
    agree = 0
    student_cost_total = 0.0
    teacher_cost_total = 0.0

    for t in shadow_tickets:
        msg = [{"role": "user", "content": TEMPLATE.format(t=t)}]
        try:
            s = ot.completion(model="ticket-classifier", messages=msg, max_tokens=10, temperature=0)
            teacher = ot.completion(model="openai/gpt-4o", messages=msg, max_tokens=10, temperature=0)
        except Exception as e:
            print(f"  skipped '{t[:40]}': {type(e).__name__}: {str(e)[:80]}")
            continue

        s_label = s.choices[0].message.content.strip().lower()
        t_label = teacher.choices[0].message.content.strip().lower()
        if s_label == t_label:
            agree += 1
        student_cost_total += s._cost or 0
        teacher_cost_total += teacher._cost or 0

    print(f"agreement:       {agree}/{len(shadow_tickets)}  ({100*agree/len(shadow_tickets):.0f}%)")
    print(f"student cost:    ${student_cost_total:.6f}")
    print(f"teacher cost:    ${teacher_cost_total:.6f}")
    if teacher_cost_total > 0:
        print(f"savings:         {100*(teacher_cost_total - student_cost_total)/teacher_cost_total:.0f}%")


**Rules of thumb:**

| Agreement | Action |
| --- | --- |
| ≥ 95% | Stay on student |
| 90–95% | Monitor; review the disagreements manually |
| 80–90% | Shadow the teacher in parallel; plan a retrain |
| < 80% | Point the alias back to the teacher; retrain with recent traces |

Shadow sampling and drift dashboards are built into the self-hosted UI under **Evaluations → Drift**.

## Clean up (optional)

Drop the alias if you don't want `"ticket-classifier"` resolving locally anymore.

In [ ]:
# ot.unset_alias("ticket-classifier")

## The full loop, in review

You've now done every step of the OpenTracy pipeline:

1. **Quickstart** — cost/latency on every call (notebook 01)
2. **Drop-in** — zero-code OpenAI migration (notebook 02)
3. **Routing** — auto-pick a model per prompt (notebook 03)
4. **End-to-end** — measure savings on a real workload (notebook 04)
5. **Distillation** — train a custom student from your traffic (notebook 05)
6. **Inference & alias swap** — ship it (**this notebook**)

The value compounds: every cluster you distill shaves more off the cost curve, and the alias swap means the app never learns about any of it.

## Further reading

- [Self-host guide](https://docs.opentracy.ai/guides/self-host) — Docker Compose for the full stack (ClickHouse, UI, management API)
- [Auto-routing concepts](https://docs.opentracy.ai/concepts/auto-routing) — how cluster assignment + error profiles work under the hood
- [Router reference](https://docs.opentracy.ai/api-reference/router) — the explicit `Router` class for rule-based routing on top of learned routing